# Task 1: Cyclone Machine Data Analysis
Here we are going to explore 3 years of cyclone sensor data.  

**Goals:**
- Detect shutdown/idle periods in the machine  
- Group machine states into clusters  
- Find abnormal sensor behavior (anomalies)  
- Forecast temperature for short-term prediction  

All results will be saved into separate folders for plots and CSV outputs.


## Import Libraries
We start by importing all libraries.  
sns.set() makes seaborn plots prettier, and rcParams sets default figure size.


In [77]:
# We will use pandas/numpy for data handling,
# matplotlib/seaborn for plots,
# sklearn for clustering and anomaly detection,
# statsmodels for ARIMA forecasting.

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA

# Make plots look nice
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (14,6)


## Setup Folders
We organize results by saving them into two folders.  
This helps us to keep the notebook clean. 

In [78]:
#  Create folders for outputs

base_path = os.getcwd()  
plots_path = os.path.join(base_path, "plots")
outputs_path = os.path.join(base_path, "outputs")

# Create folders if not already exist
os.makedirs(plots_path, exist_ok=True)
os.makedirs(outputs_path, exist_ok=True)


## Load Dataset
We prepare the data by:
- Converting sensor readings to numeric
- Setting timestamp as index
- Keeping only the relevant sensor columns


In [79]:
# Load the cyclone dataset
file_path = os.path.join(base_path, "cyclone_data.xlsx")
df = pd.read_excel(file_path)

# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Define sensor columns to analyze
sensor_cols = [
    'Cyclone_Inlet_Gas_Temp', 'Cyclone_Gas_Outlet_Temp', 
    'Cyclone_Outlet_Gas_draft', 'Cyclone_cone_draft', 
    'Cyclone_Inlet_Draft', 'Cyclone_Material_Temp'
]

# Convert columns to numeric values
for col in sensor_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert time to datetime and set as index
df['time'] = pd.to_datetime(df['time'], errors='coerce')
df.set_index('time', inplace=True)

df_numeric = df[sensor_cols]  # this dataframe only for sensor data


c:\Users\attar\Desktop\Mahiboob_Attar_DataSceince\datascience\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## Resample & Fill Missing Data
Sensor data might have missing readings.  
We use forward and backward fill to make it continuous for analysis.

In [80]:
# Make data 5-min interval and fill missing values
df_numeric = df_numeric.resample('5min').mean()  # average per 5 minutes
df_numeric.ffill(inplace=True)  # fill missing forward
df_numeric.bfill(inplace=True)  # fill remaining backward


## Handling Extreme Outliers

Sensor data can have extreme values (spikes or dips) which may distort analysis, clustering, and forecasting.  
We detect outliers using the **IQR (Interquartile Range) method**: values below Q1 - 1.5*IQR or above Q3 + 1.5*IQR are considered extreme.

Two ways to handle outliers:

1. **Median Replacement**  
   - Replace outliers with the **median** of the column.  
   - **Reasoning:** Strongly reduces the impact of outliers. Best for anomaly-sensitive analysis where extreme spikes may give wrong signals.  
   - Preserves central tendency but removes extreme distortions.

2. **Clipping using np.clip**  
   - Limit values to the IQR bounds. Values below the lower bound are set to the lower bound, and values above the upper bound are set to the upper bound.  
   - **Reasoning:** Keeps data more realistic, preserves trends, and is faster for large datasets.  
   - Useful when we want to reduce extreme spikes but maintain overall data distribution.



In [81]:
# Remove extreme outliers using IQR and replace with median
def replace_outliers_with_median(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    
    median_val = data[col].median()
    data[col] = np.where(data[col] < lower, median_val, data[col])
    data[col] = np.where(data[col] > upper, median_val, data[col])

for col in df_numeric.columns:
    replace_outliers_with_median(df_numeric, col)
    
# Ensure any remaining NaNs are filled with median (prevents clustering errors)
df_numeric.fillna(df_numeric.median(), inplace=True)

_ = """

# Alternative: Clip extreme values using IQR bounds
def clip_outliers(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    
    data[col] = np.clip(data[col], lower, upper)

for col in df_numeric.columns:
    clip_outliers(df_numeric, col)
    
"""

## Exploratory Analysis
We check basic statistics and visualize the data.  
- Heatmap shows how sensors correlate  
- Weekly and yearly plots show sensor trends over time


In [82]:
#  Basic analysis and plots

# Save summary stats to CSV
df_numeric.describe().to_csv(os.path.join(outputs_path, "summary_statistics.csv"))

# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df_numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "correlation_matrix.png"))
plt.close()

# Plot 1 week data
df_numeric['2017-01-01':'2017-01-07'].plot(title="Cyclone Sensor Data: 1 Week")
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "one_week.png"))
plt.close()

# Plot 1 year data
df_numeric['2017-01-01':'2017-12-31'].plot(title="Cyclone Sensor Data: 1 Year")
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "one_year.png"))
plt.close()


## Detect Shutdown Periods
Shutdown detection highlights periods when most sensors were very low (below 5th percentile).  
The plot shows shutdown periods highlighted in red.
### Shutdown Detection
- Uses **sensor-specific percentiles** instead of fixed thresholds.
- Avoids false negatives for idle periods.
- Option to ignore **short shutdowns (<10 min)** to remove unnecessary noise.

### Shutdown Detection Threshold Comparison

To detect machine shutdowns or idle periods, we examine when most sensors fall below their 5th percentile value.  
We compare two thresholds:

- **80% threshold (`>0.8`)**: Strict shutdown detection. Only periods where nearly all sensors are low are considered shutdowns.  
- **60% threshold (`>0.6`)**: Less strict, captures more idle or partial low-activity periods.  

This comparison helps decide the most meaningful shutdown detection for analysis and downstream tasks.
- **Red areas** show strict shutdowns (`>0.8` sensors low).  
- **Orange areas** show more relaxed detection (`>0.6` sensors low).  
- The 80% threshold captures **only true shutdowns**, while the 60% threshold also captures **partial idle periods**.  
- For downstream clustering and anomaly detection, we recommend using **80% threshold** to ensure active periods are truly operational.



In [83]:
# Compute thresholds (5th percentile per sensor)
thresholds = df_numeric.quantile(0.05)

# Shutdown mask for strict detection (80%)
shutdown_mask_80 = (df_numeric < thresholds).mean(axis=1) > 0.8

# Shutdown mask for relaxed detection (60%)
shutdown_mask_60 = (df_numeric < thresholds).mean(axis=1) > 0.6

# Generate shutdown periods dataframes
def get_shutdown_periods(mask):
    shutdowns = []
    in_shutdown = False
    for time, is_down in mask.items():
        if is_down and not in_shutdown:
            start_time = time
            in_shutdown = True
        elif not is_down and in_shutdown:
            end_time = time
            shutdowns.append({
                'start': start_time,
                'end': end_time,
                'duration_min': (end_time-start_time).total_seconds()/60
            })
            in_shutdown = False
    if in_shutdown:
        end_time = mask.index[-1]
        shutdowns.append({
            'start': start_time,
            'end': end_time,
            'duration_min': (end_time-start_time).total_seconds()/60
        })
    return pd.DataFrame(shutdowns)

shutdown_df_80 = get_shutdown_periods(shutdown_mask_80)
shutdown_df_60 = get_shutdown_periods(shutdown_mask_60)

# Save CSVs
shutdown_df_80.to_csv(os.path.join(outputs_path, "shutdown_periods_80.csv"), index=False)
shutdown_df_60.to_csv(os.path.join(outputs_path, "shutdown_periods_60.csv"), index=False)

# Compute total shutdown events and downtime
num_shutdowns_80 = len(shutdown_df_80)
total_downtime_80 = shutdown_df_80['duration_min'].sum()
num_shutdowns_60 = len(shutdown_df_60)
total_downtime_60 = shutdown_df_60['duration_min'].sum()

# Plot comparison for one year and save
plt.figure(figsize=(14,6))
plt.plot(df_numeric['2017-01-01':'2017-12-31']['Cyclone_Inlet_Gas_Temp'], 
         label='Cyclone_Inlet_Gas_Temp', color='blue')

# Plot 80% threshold shutdowns in red
for _, row in shutdown_df_80.iterrows():
    plt.axvspan(row['start'], row['end'], color='red', alpha=0.3, 
                label='Shutdown 80%' if _==0 else "")

# Plot 60% threshold shutdowns in orange
for _, row in shutdown_df_60.iterrows():
    plt.axvspan(row['start'], row['end'], color='orange', alpha=0.2, 
                label='Shutdown 60%' if _==0 else "")

plt.title(f"Shutdown Detection Comparison: 80% vs 60% Thresholds\n"
          f"80% → {num_shutdowns_80} events, {total_downtime_80:.1f} min total | "
          f"60% → {num_shutdowns_60} events, {total_downtime_60:.1f} min total",
          fontsize=12)
plt.xlabel("Time")
plt.ylabel("Cyclone Inlet Gas Temp")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "shutdown_threshold_comparison.png"))
plt.close()


## Machine State Clustering
Clustering helps to understand different operating modes of the machine.
- Cluster operating states of the machine using KMeans.
- Only use data outside shutdown periods.
- Each cluster represents a different machine state.
- We cluster active machine states into 4 groups (Normal, Startup/Shutdown, High Load, Degraded) for analysis.



In [84]:
# Align the mask with the DataFrame index
aligned_mask = shutdown_mask.reindex(df_numeric.index, fill_value=False)

# Select only active data (not shutdown)
active_data = df_numeric[~aligned_mask].copy()

# Apply KMeans clustering
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
active_data.loc[:, 'cluster'] = kmeans.fit_predict(active_data)


# Save cluster summary
cluster_summary = active_data.groupby('cluster').agg(['mean', 'std', 'count'])
cluster_summary.to_csv(os.path.join(outputs_path, "clusters_summary.csv"))


## Contextual Anomaly Detection
Detect anomalies within each cluster using Isolation Forest.
- Only 1% contamination assumed.
- Saves anomalous periods with sensor readings.

In [85]:
anomalies_list = []
for cluster_id in active_data['cluster'].unique():
    cluster_data = active_data[active_data['cluster']==cluster_id].copy()
    iso = IsolationForest(contamination=0.01, random_state=42)
    cluster_data['anomaly'] = iso.fit_predict(cluster_data[sensor_cols])
    
    # Record anomalies
    cluster_anomalies = cluster_data[cluster_data['anomaly']==-1]
    for idx, row in cluster_anomalies.iterrows():
        anomalies_list.append({
            'time': idx,
            'cluster': cluster_id,
            **row[sensor_cols].to_dict()
        })

anomalies_df = pd.DataFrame(anomalies_list)
anomalies_df.to_csv(os.path.join(outputs_path, "anomalous_periods.csv"), index=False)

## Short-Horizon Forecasting
We forecast the Cyclone Inlet Gas Temp for the next hour using:
- Persistence model (baseline)
- ARIMA model

The plot compares predicted vs true values.


In [86]:

# Prepare train and test sets
y = df_numeric['Cyclone_Inlet_Gas_Temp']
train_size = int(0.8 * len(y))
train, test = y[:train_size], y[train_size:]

# Persistence model (baseline)

# This simply predicts the previous value as the next value
pred_persist = test.shift(1).bfill()  # fill NaNs with next valid value

rmse_persist = np.sqrt(mean_squared_error(test, pred_persist))
mae_persist = mean_absolute_error(test, pred_persist)

# ARIMA model for forecasting
model = ARIMA(train, order=(5,1,0))
model_fit = model.fit()
pred_arima = model_fit.forecast(len(test))

rmse_arima = np.sqrt(mean_squared_error(test, pred_arima))
mae_arima = mean_absolute_error(test, pred_arima)

print(f"Persistence RMSE: {rmse_persist:.2f} MAE: {mae_persist:.2f}")
print(f"ARIMA RMSE: {rmse_arima:.2f} MAE: {mae_arima:.2f}")


# Save forecasts to CSV
forecasts_df = pd.DataFrame({
    'true': test,
    'persist_pred': pred_persist,
    'arima_pred': pred_arima
})
forecasts_df.to_csv(os.path.join(outputs_path, "forecasts.csv"))

# Plot first 200 points for comparison
plt.figure(figsize=(14,6))
plt.plot(test.index[:200], test.values[:200], label='True')
plt.plot(test.index[:200], pred_persist[:200], label='Persistence')
plt.plot(test.index[:200], pred_arima[:200], label='ARIMA')
plt.title("Cyclone_Inlet_Gas_Temp Forecast Comparison")
plt.xlabel("Time")
plt.ylabel("Temperature")
plt.legend()
plt.tight_layout()
plt.close()


Persistence RMSE: 15.82 MAE: 9.74
ARIMA RMSE: 17.60 MAE: 13.49


## Completion Message
All CSV files and plots are saved for inspection. Task 1 analysis is complete.


In [87]:
print("All outputs saved in 'outputs/' and plots in 'plots/'")


All outputs saved in 'outputs/' and plots in 'plots/'
